# Lab 1：静态页面与 API 数据采集

- 平台：SmartPark DataLab（`https://115.159.159.82`）
- Device：动态加载 → API → JSON → 分页采集全部设备（5000 条）
- Company：静态加载 → HTML → BeautifulSoup → “下一页” → 采集 3 页（60 条）

## 0. 环境与公共配置

In [1]:
import json
from collections import Counter
from pathlib import Path
from urllib.parse import urljoin

import requests
import urllib3
from bs4 import BeautifulSoup

urllib3.disable_warnings()

BASE = "https://115.159.159.82"
s = requests.Session()
s.verify = False  # 课程服务器为自签名证书

## 1. 任务一：Device 动态加载 —— 找到 API

在 `/devices` 页面“查看网页源代码”中搜不到设备名称，说明设备数据由 JS 动态加载。
打开 `F12 → Network → Fetch/XHR`，刷新页面后可看到请求：

- Request URL：`https://115.159.159.82/api/devices`
- Request Method：`GET`
- Query 参数：`page=1`、`page_size=50`

下面先请求第一页，观察 JSON 结构。

In [2]:
api_url = BASE + "/api/devices"

r = s.get(api_url, params={"page": 1, "page_size": 50}, timeout=20)
print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

data = r.json()
print("type:", type(data))
print("keys:", list(data.keys()))
print("total:", data["total"], "| page_size:", data["page_size"], "| page:", data["page"])

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json
type: <class 'dict'>
keys: ['page', 'page_size', 'total', 'items']
total: 5000 | page_size: 50 | page: 1


JSON 结构字段含义：`page` 当前页、`page_size` 每页条数、`total` 设备总数、`items` 本页设备列表。

In [3]:
print(data["items"][0])

{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


## 2. 任务一：根据 total / page_size 自动计算总页数并采集全部设备

In [4]:
total = data["total"]
page_size = data["page_size"]
total_pages = (total + page_size - 1) // page_size
print("total:", total, "| page_size:", page_size, "| total_pages:", total_pages)

devices = []
for page in range(1, total_pages + 1):
    r = s.get(api_url, params={"page": page, "page_size": page_size}, timeout=20)
    r.raise_for_status()
    devices.extend(r.json()["items"])

print("device count:", len(devices))

total: 5000 | page_size: 50 | total_pages: 100


device count: 5000


## 3. 任务二：Company 静态加载判断

`/companies` 页面“查看网页源代码”中能搜到企业名称，说明企业数据随 HTML 一起返回，按静态页面处理。

In [5]:
base_url = BASE
current_url = base_url + "/companies"

r = s.get(current_url, timeout=20)
soup = BeautifulSoup(r.text, "html.parser")
print("status:", r.status_code)

status: 200


企业记录最外层节点是 `<article class="company-card" data-company-id="C000001">`。
selector 优先使用语义明确、稳定的 `class`，取 `article.company-card`。

In [6]:
cards = soup.select("article.company-card")
print("本页企业数：", len(cards))

本页企业数： 20


## 4. 任务二：提取企业字段

先解析第一条记录；行业、区域、员工规模、设备数量是 `dt/dd` 结构，先转成字典再取值。

In [7]:
card = cards[0]
details = {}
for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

print(details)

{'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}


In [8]:
def parse_company(card):
    details = {}
    for item in card.select("dl > div"):
        key = item.select_one("dt").get_text(strip=True)
        value = item.select_one("dd").get_text(strip=True)
        details[key] = value
    return {
        "company_id": card.get("data-company-id"),
        "name": card.select_one("h2 a").get_text(strip=True),
        "industry": details["所属行业"],
        "zone": details["所在区域"],
        "device_count": int(details["设备数量"].split()[0]),
        "status": card.select_one(".company-card-top .badge").get_text(strip=True),
    }


print(parse_company(cards[0]))

{'company_id': 'C000001', 'name': '临港精工001有限公司', 'industry': '汽车零部件', 'zone': '智造一区', 'device_count': 6, 'status': '运营中'}


## 5. 任务三：根据“下一页”连续采集 3 页

分页栏结构为 `<nav class="pagination">`，其中“下一页”是最后一个 `<a>`，
可用 `nav.pagination a:last-child` 稳定定位。

In [9]:
next_link = soup.select_one("nav.pagination a:last-child")
print(next_link.get_text(" ", strip=True))
print(next_link.get("href"))

下一页
/companies?page=2


In [10]:
companies = []
current_url = base_url + "/companies"

for _ in range(3):
    r = s.get(current_url, timeout=20)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    for card in soup.select("article.company-card"):
        companies.append(parse_company(card))
    next_link = soup.select_one("nav.pagination a:last-child")
    current_url = urljoin(current_url, next_link["href"])

print("company count:", len(companies))

company count: 60


## 6. 任务四：验证采集结果

In [11]:
print("companies:", len(companies))
print("unique company_id:", len(set(x["company_id"] for x in companies)))
print("devices:", len(devices))
print("unique device_id:", len(set(x["device_id"] for x in devices)))

device_counts = Counter(x["company_id"] for x in devices)
mismatch_count = 0
for company in companies:
    cid = company["company_id"]
    if company["device_count"] != device_counts.get(cid, 0):
        mismatch_count += 1

print("Company–Device 不一致企业数：", mismatch_count)

companies: 60
unique company_id: 60
devices: 5000
unique device_id: 5000
Company–Device 不一致企业数： 0


## 7. 任务五：保存采集结果

In [12]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")

数据已保存到 data/ 目录
